In [0]:
USE CATALOG workspace;
USE SCHEMA causal_project;

In [0]:
CREATE OR REPLACE TABLE gold_household
USING DELTA
AS
WITH pre AS (
  SELECT
    household_key,
    campaign_start,
    COUNT(DISTINCT week_no)                               AS pre_active_weeks,
    COUNT(DISTINCT basket_id)                             AS pre_total_trips,
    COUNT(DISTINCT department)                            AS pre_department_breadth,
    SUM(customer_spend)                                   AS pre_total_spend,
    SUM(coupon_used)                                      AS pre_total_coupon_use,
    SUM(loyalty_card_used)                                AS pre_total_loyalty_use,
    SUM(customer_spend)    / COUNT(DISTINCT week_no)      AS pre_avg_weekly_spend,
    COUNT(DISTINCT basket_id) / COUNT(DISTINCT week_no)   AS pre_avg_weekly_trips,
    SUM(coupon_used)       / COUNT(DISTINCT week_no)      AS pre_coupon_usage_rate,
    SUM(loyalty_card_used) / COUNT(DISTINCT week_no)      AS pre_loyalty_card_rate
  FROM silver_transactions
  WHERE pre_campaign = 1
  GROUP BY household_key, campaign_start
),
during AS (
  SELECT
    household_key,
    SUM(customer_spend)                                 AS total_campaign_spend,
    COUNT(DISTINCT basket_id)                       AS total_campaign_trips,
    SUM(customer_spend) / MAX(clean_window_days)         AS avg_daily_campaign_spend,
    COUNT(DISTINCT basket_id) / MAX(clean_window_days)    AS avg_daily_campaign_trips
  FROM silver_transactions
  WHERE in_campaign_window = 1
  GROUP BY household_key
),
household_attrs AS (
  SELECT
    household_key,
    MAX(treatment)          AS treatment,
    MAX(clean_window_days)  AS clean_window_days,
    MAX(classification_1)   AS age_group,
    MAX(classification_2)   AS marital_status,
    MAX(classification_3)   AS income_level,
    MAX(classification_4)   AS household_size,
    MAX(homeowner_desc)     AS home_ownership,
    MAX(kid_category_desc)  AS kid_count
  FROM silver_transactions
  GROUP BY household_key
)

SELECT
  -- Identifier
  h.household_key,

  -- Treatment
  h.treatment,

 -- Primary outcome 
  COALESCE(d.avg_daily_campaign_spend, 0)  AS avg_daily_campaign_spend,

  -- Secondary outcomes
  COALESCE(d.total_campaign_spend, 0)      AS total_campaign_spend,
  COALESCE(d.avg_daily_campaign_trips, 0)  AS avg_daily_campaign_trips,

  -- Pre-campaign confounders
  p.pre_avg_weekly_spend,
  p.pre_avg_weekly_trips,
  p.pre_coupon_usage_rate,
  p.pre_loyalty_card_rate,
  p.pre_department_breadth,

  -- Technical controls 
  p.pre_active_weeks,
  h.clean_window_days,

  -- Demographic effect modifiers 
  h.age_group,
  h.marital_status,
  h.income_level,
  h.household_size,
  h.home_ownership,
  h.kid_count

FROM household_attrs h
JOIN pre    p ON h.household_key = p.household_key
LEFT JOIN during d ON h.household_key = d.household_key

### Zero-Outcome Households
14 of 760 households received a campaign but made no purchases 
during their campaign window (avg_daily_campaign_spend = 0). 
A household that received a campaign and did not respond is 
meaningful signal, not missing data. Dropping them would 
upward-bias the ATE estimate.